# LAGN XGBoost Classifier

*Notebook written by Zach Gillis (zgillis@stanford.edu)*

*Date last updated: March 9, 2026*

This is the last of three notebooks to prepare the *First Cut* filter to select lensed-AGN candidates from the Rubin Observatory alert stream using difference image analysis sources (DIASources).

The previous notebook (`2_inj_source_characterization.ipynb`) built the labeled training dataset by cross-matching Shenming's injected LAGN DIASources against injection coordinates and combining them with non-LAGN DP1 DIASources.

This notebook trains an XGBoost binary classifier to distinguish LAGN DIASources from non-LAGN DIASources, evaluates it on a held-out test set, prepares a completeness/purity curve using estimated LSST survey statistics, and compares the ROC curves of the XGBoost binary classifier with the simple cuts filter. 

## Overview

1. **Load & split data** — load `combined_training_data.csv`, select features, and split into stratified train/test sets
2. **Hyperparameter search** — 5-fold stratified CV with `GridSearchCV` over key XGBoost parameters, scored by ROC-AUC, which is invariant to class ratio
3. **Train final model** — fit XGBoost with best parameters
4. **Evaluate** — classification metrics and diagnostic plots on the held-out test set
5. **Survey scale completeness/purity analysis** — rescale classifier performance to the true LSST class ratio (~1:37,000 LAGN:non-LAGN)
6. **Simple cuts comparison** — compare XGBoost classifier against the simple cuts filter by sweeping the `i_ext` threshold
7. **Validation on uninjected data** — apply the classifier to DIASources from visits with no injected sources to assess effects of potential Rubin data processing pipeline differences

---
## Section 1 — Setup & Data Loading

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yaml

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, auc, confusion_matrix,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
)
from sklearn.calibration import calibration_curve

### Load & Split Data

Loads `combined_training_data.csv` (produced in the previous notebook) — which contains one row per DIASource with `label=0` (non-LAGN DIASource, DP1) or `label=1` (LAGN DIASource, injected).

A subset of features to use for training is selected from the candidate features included in `combined_training_data.csv`.

The dataset is split 80% train / 20% test with `stratify=y` to preserve the class ratio in both splits. `scale_pos_weight` is set to the negative/positive count ratio in the training set to account for the class imbalance. 

XGBoost natively supports NaN values (present in dipole-specific columns in DIASources not flagged as dipoles) and categorical variables (i.e. band).

**Features used:** `trailLength`, `temp_sci_flux_ratio`, `i_ext`, `flux_ext`, `ellip_ext`, `dipoleLength`, `band`, `template_flux`, `scienceFlux`, `psfFlux`, `dipoleMeanFlux`, `apFlux`

In [ ]:
FEATURES = [
    'trailLength', 'temp_sci_flux_ratio', 'i_ext', 'flux_ext', 'ellip_ext',
    'dipoleLength', 'band', 'template_flux', 'scienceFlux', 'psfFlux',
    'dipoleMeanFlux', 'apFlux',
]

df = pd.read_csv('combined_training_data.csv')

df['isDipole'] = df['isDipole'] == 'True'
df['band'] = pd.Categorical(df['band'])
band_categories = df['band'].cat.categories

X = df.drop(columns=['label'])[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"Training samples     : {len(X_train):,}")
print(f"Test samples         : {len(X_test):,}")
print(f"neg/pos              : {scale_pos_weight:.1f}")

---
## Section 2 — Hyperparameter Search

### Grid Search with 5-Fold Cross-Validation

Searches over a grid of XGBoost hyperparameters using `GridSearchCV` with 5-fold stratified cross-validation, scored by ROC-AUC.

ROC-AUC is used because it is invariant to class ratio — this data has a about 1 lens : 50 non-lens ratio compared to the expected survey 1 lens : 37,000 non-lens ratio.

In [ ]:
param_grid = {
    'max_depth':        [4, 6, 8],
    'learning_rate':    [0.01, 0.02, 0.03],
    'subsample':        [0.6, 0.8],
    'colsample_bytree': [0.7, 0.9],
}

search = GridSearchCV(
    XGBClassifier(
        n_estimators=500,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    refit=False,
    verbose=1,
)
search.fit(X_train, y_train)

best_params = search.best_params_
print(f"\nBest params: {best_params}")

---
## Section 3 — Train Final Model

### Fit XGBoost with Early Stopping

Trains the final classifier using the best hyperparameters from the grid search (or set `USE_BEST_PARAMS = False` if you'd like to set your own parameters manually).

In [ ]:
# ── Train Final Model ──────────────────────────────────────────────────────────
# Set USE_BEST_PARAMS = False to train with the default hyperparameters instead.
USE_BEST_PARAMS = True

MANUAL_PARAMS = {
    'max_depth':        6,
    'learning_rate':    0.03,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

model = XGBClassifier(
    n_estimators=1000,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(f"Training complete. Best n_estimators: {model.best_iteration}")

---
## Section 4 — Evaluation on Held-Out Test Set

### Test Set Metrics

Evaluates the trained model on `X_test` and `y_test`, the held-out test set from earlier. 

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-LAGN (0)', 'LAGN (1)'], digits=4))

### Visualization: Diagnostic Plots

Five diagnostic plots summarizing classifier performance on the held-out test set:

| Plot | Description |
|---|---|
| **Feature Importance** | XGBoost gain-based importance for each input feature |
| **ROC Curve** | True positive rate vs. false positive rate across all thresholds |
| **Precision-Recall Curve** | Precision vs. recall at the training-set class ratio (~1:50) |
| **P(LAGN) Distribution** | Predicted probability histograms for LAGN and non-LAGN DIASources |
| **Confusion Matrix** | Counts and row-normalized rates at nominal threshold (0.5) |

In [ ]:
fig = plt.figure(figsize=(15, 10), dpi=300)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# Feature Importance
ax1 = fig.add_subplot(gs[0, 0])
importance = model.feature_importances_
feat_names = np.array(X.columns.tolist())
top_idx = np.argsort(importance)[-20:]
ax1.barh(feat_names[top_idx], importance[top_idx], color='steelblue', edgecolor='white')
ax1.set_xlabel('Gain', fontsize=11)
ax1.set_title('Feature Importance', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelsize=8)
ax1.spines[['top', 'right']].set_visible(False)

# ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost (AUC = {roc_auc_val:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random')
ax2.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.02])
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.spines[['top', 'right']].set_visible(False)

# Precision-Recall Curve
ax3 = fig.add_subplot(gs[0, 2])
pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
baseline = y_test.mean()
ax3.plot(pr_recall, pr_precision, color='darkorange', lw=2, label=f'XGBoost (AP = {ap:.4f})')
ax3.axhline(baseline, color='navy', lw=1.5, linestyle='--',
            label=f'Random (AP = {baseline:.4f})')
ax3.fill_between(pr_recall, pr_precision, alpha=0.08, color='darkorange')
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.02])
ax3.set_xlabel('Recall', fontsize=11)
ax3.set_ylabel('Precision', fontsize=11)
ax3.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.spines[['top', 'right']].set_visible(False)

# Predicted Probability Distribution
ax4 = fig.add_subplot(gs[1, 0])
bins = np.linspace(0, 1, 60)
ax4.hist(y_prob[y_test == 0], bins=bins, alpha=0.65, color='steelblue',
         label='Non-LAGN (0)', density=True)
ax4.hist(y_prob[y_test == 1], bins=bins, alpha=0.65, color='darkorange',
         label='LAGN (1)', density=True)
ax4.axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax4.set_xlabel('P(LAGN)', fontsize=11)
ax4.set_ylabel('Density', fontsize=11)
ax4.set_title('P(LAGN)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.spines[['top', 'right']].set_visible(False)

# Confusion Matrix Heatmap
ax5 = fig.add_subplot(gs[1, 1])
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels_raw = [f'{v:,}' for v in cm.ravel()]
labels_pct = [f'{v:.1%}' for v in cm_norm.ravel()]
annot = np.array([f'{r}\n({p})' for r, p in zip(labels_raw, labels_pct)]).reshape(2, 2)

sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax5,
            xticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            yticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Row-normalised rate'})
ax5.set_xlabel('Predicted Label', fontsize=11)
ax5.set_ylabel('True Label', fontsize=11)
ax5.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.show()

---
## Section 5 — Survey Scale Analysis

The test set metrics above reflect a ~1:50 class ratio, not the LSST DIASource LAGN to non-LAGN ratio of ~1:37,000. At survey scale, the same classifier will produce far more false positives relative to true positives.

Here we use only invariant quantities like true positive rate (completeness, TPR) and false positive rate (FPR) to prepare a representative completeness/purity curve.

## Survey Population Estimates

We estimate the number of LAGN and non-LAGN DIASources expected in the 10-year LSST survey and in year 1. These are rough calculations.

### LAGN DIASources

| Parameter | Value |
|---|---|
| Expected lensed AGN in full survey | 4,000 |
| Visits per sky position (over 10 years) | 800 |
| LAGN detection rate (DIASource per visit per AGN) | 10% |
| **Total LAGN DIASources (10 yr)** | **320,000** |
| **Total LAGN DIASources (yr 1, 80 visits)** | **32,000** |

### Non-LAGN DIASources

Non-LAGN DIASources are estimated from the LSST DP1 ECDFS (within the wide fast deep survey area).

| Parameter | Value |
|---|---|
| DP1 DIASources (ECDFS) | 551,975 |
| DP1 visits | 855 |
| DP1 field area | 0.785 deg² |
| DIASources / visit / deg² | 822.4 / visit / deg² |
| Full survey area | 18,000 deg² |
| **Total background DIASources (10 yr)** | **~11.8 billion** |
| **Total background DIASources (yr 1)** | **~1.18 billion** |


The ratio of non-LAGN to LAGN DIASources is **~37,000:1**.

### Completeness–Purity Analysis

We sweep the classification threshold and compute the completeness and purity at survey scale using 10-fold stratified cross-validation (for smoothness).

Completeness (TPR) and FPR are can be measured on the training data and then rescaled to yield counts of true positives and false positives:

$$N_\text{LAGN,yr1} = 32,000$$
$$N_\text{non-LAGN,yr1} = 1,184,257,459$$

$$\text{TP} = \text{TPR} \times N_\text{LAGN,yr1}$$
$$\text{FP} = \text{FPR} \times N_\text{non-LAGN,yr1}$$
$$\text{purity} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

The purity is averaged across all 10 folds. Two plots are produced:
1. **Completeness vs. Purity**
2. **LAGN DIASources vs. Purity** (how many LAGN DIASources detected at each purity level in year 1)

In [ ]:
lagn_diasources     = 32_000
non_lagn_diasources = 1_184_257_459

print(f"LAGN DIASources     (yr 1) : {lagn_diasources:,}")
print(f"Non-LAGN DIASources (yr 1) : {non_lagn_diasources:,}")
print(f"Non-LAGN:LAGN ratio        : 1:{non_lagn_diasources // lagn_diasources:,}")

cv          = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
comp_grid   = np.linspace(0, 1, 100)
fold_purity = []

for train_idx, val_idx in cv.split(X, y):
    fold_model = XGBClassifier(
        n_estimators=300, scale_pos_weight=neg / pos,
        eval_metric='logloss', enable_categorical=True,
        random_state=42, n_jobs=-1, **params,
    )
    fold_model.fit(X.iloc[train_idx], y.iloc[train_idx], verbose=False)
    
    # Get predicted probabilities and true labels for validation fold
    prob   = fold_model.predict_proba(X.iloc[val_idx])[:, 1]
    labels = np.array(y.iloc[val_idx])

    # Sort validation DIASources by descending predicted probability
    order  = np.argsort(prob)[::-1]
    labels = labels[order]

    # Total positives and negatives in this fold
    N_pos = (labels == 1).sum()
    N_neg = (labels == 0).sum()

    # Cumulative TP, FP, TN, FN as we descend the sorted validation DIASources
    TP = np.cumsum(labels == 1)
    FP = np.cumsum(labels == 0)

    # Calculate TPR and FPR as we descend the sorted validation DIASources
    tpr = TP / N_pos
    fpr = FP / N_neg

    # Scale to survey population to get true purity
    TP_survey = tpr * lagn_diasources
    FP_survey = fpr * non_lagn_diasources
    purity    = TP_survey / (TP_survey + FP_survey).clip(1e-12)

    fold_purity.append(np.interp(comp_grid, tpr, purity))

mean_purity     = np.mean(fold_purity, axis=0)
sample_size_yr1 = comp_grid * lagn_diasources

# Plot 1: Completeness vs. Purity
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
ax.plot(comp_grid, mean_purity, color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(0, 1)
ax.set(xlabel='Completeness', ylabel='Purity',
       title='Completeness–Purity (10-fold CV)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Plot 2: LAGN DIASources vs. Purity
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
ax.plot(mean_purity, sample_size_yr1, color='darkorange', lw=2)
ax.set_xscale('log')
ax.set_xlim(1e-4, 1)
ax.set(xlabel='Purity', ylabel='LAGN DIASources (year 1)',
       title='LAGN DIASources vs. Purity (year 1)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### XGBoost vs. Simple Cuts Filter

Compares ROC curves for XGBoost and the simple cuts filter. Three curves are shown:

- **XGBoost binary classifier**
- **All cuts, sweep `i_ext`** — holds `flux_ext`, `ellip_ext`, and `temp_sci_flux_ratio` at their fixed thresholds and sweeps `i_ext`. The line doesn't reach a completeness of one, as the other cuts remove LAGN DIASources before we even sweep `i_ext`
- **`i_ext` only** — sweeps `i_ext` alone with no other cuts applied

In [ ]:
import yaml

with open('simple_cuts_thresholds.yaml') as f:
    sc = yaml.safe_load(f)

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob)
auc_xgb = roc_auc_score(y_test, y_prob)

# i_ext only ROC
valid = X_test['i_ext'].notna()
fpr_iext, tpr_iext, _ = roc_curve(y_test[valid], X_test.loc[valid, 'i_ext'])
auc_iext = roc_auc_score(y_test[valid], X_test.loc[valid, 'i_ext'])

# Full simple cuts ROC
flux_cap_array = np.array([sc['flux_caps'][band] for band in X_test['band']])
passes_other = (
    (X_test['flux_ext']            > sc['flux_ext'])           &
    (X_test['ellip_ext']           > sc['ellip_ext'])          &
    (X_test['temp_sci_flux_ratio'] > sc['temp_sci_flux_ratio']) &
    (X_test['template_flux']       < flux_cap_array)           &
    X_test['i_ext'].notna()
)

i_ext_sub  = X_test.loc[passes_other, 'i_ext'].values
y_sub      = y_test[passes_other].values
N_pos_full = (y_test == 1).sum()
N_neg_full = (y_test == 0).sum()

order      = np.argsort(i_ext_sub)[::-1]
labels_ord = y_sub[order]
TP = np.cumsum(labels_ord == 1)
FP = np.cumsum(labels_ord == 0)
tpr_sc = np.concatenate([[0], TP / N_pos_full])
fpr_sc = np.concatenate([[0], FP / N_neg_full])

# Plot
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

ax.plot(fpr_xgb,  tpr_xgb,  color='darkorange', lw=2, label=f'XGBoost (AUC = {auc_xgb:.4f})')
ax.plot(fpr_iext, tpr_iext, color='steelblue',  lw=2, label=f'i_ext only (AUC = {auc_iext:.4f})')
ax.plot(fpr_sc,   tpr_sc,   color='steelblue',  lw=2, ls='--', label='All cuts, sweep i_ext')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


---
## Section 6 — Validation on Uninjected Data within Injection Pipeline)

Applies the trained classifier to DIASources from visits in the injection pipeline where no sources were actually injected.

This serves as a sanity check to see whether the classifier is detecting differences in the data pipelines (DP1 uses an older difference image analysis pipeline).

In [ ]:
# file created in notebook 2
noinj = pd.read_csv('diasources_no_inj.csv')

noinj['isDipole'] = noinj['isDipole'] == 'True'
noinj['band'] = pd.Categorical(noinj['band'], categories=band_categories)

X_noinj = noinj[X.columns]

noinj_pred = model.predict(X_noinj)
noinj_prob = model.predict_proba(X_noinj)[:, 1]

threshold = 0.5

fpr_test   = (y_pred[y_test == 0] == 1).mean()
fpr_noinj  = noinj_pred.mean()

print(f"FPR on held-out test set : {fpr_test:.4f}  ({fpr_test*100:.2f}%)")
print(f"FPR on uninjected data   : {fpr_noinj:.4f}  ({fpr_noinj*100:.2f}%)")

prob_bg_test = y_prob[y_test == 0]

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
bins = np.linspace(0, 1, 60)

ax.hist(prob_bg_test, bins=bins, density=True, alpha=0.6,
        color='steelblue', label=f'Non-LAGN test set  (FPR = {fpr_test:.3f})')
ax.hist(noinj_prob, bins=bins, density=True, alpha=0.6,
        color='darkorange', label=f'Uninjected data  (FPR = {fpr_noinj:.3f})')
ax.axvline(threshold, color='red', lw=1.5, ls='--', label=f'Threshold = {threshold}')

ax.set_xlabel('P(LAGN)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
